# Pivot Tables

A pivot table is a way of summarizing data in a DataFrame for a particular purpose. It makes heavy use of the aggregation function. A pivot table is itself a DataFrame, where the rows represent one variable that you're interested in, the columns another, and the cell's some aggregate value.

In [9]:
import pandas as pd
import numpy as np

### Loading Data

Let's load the Center for World University Rankings (CWUR) dataset to practice pivot tables.

In [10]:
df = pd.read_csv("datasets/cwurData.csv")

df.head()

,world_rank,institution,country,national_rank,quality_of_education,alumni_employment,quality_of_faculty,publications,influence,citations,broad_impact,patents,score,year
0,1,Harvard University,USA,1,7,9,1,1,1,1,NaN,5,100.00,2012
1,2,Massachusetts Institute of Technology,USA,2,9,17,3,12,4,4,NaN,1,91.67,2012
2,3,Stanford University,USA,3,17,11,5,4,2,2,NaN,15,89.50,2012
3,4,University of Cambridge,United Kingdom,1,10,24,4,16,16,11,NaN,50,86.17,2012
4,5,California Institute of Technology,USA,4,2,29,7,37,22,22,NaN,18,85.21,2012


First, let's create a new categorical column `Rank_Level` mapping the numeric world rank into tiers (First Tier, Second Tier, etc.).

In [11]:
def createCateg(ranking):
    if (ranking >= 1 and ranking <= 100):
        return "First Tier Top University"
    elif (ranking > 100 and ranking <= 200):
        return "Second Tier Top University"
    elif (ranking > 200 and ranking <= 300):
        return "Third Tier Top University"
    else:
        return "Other Top University"
    
df["Rank_Level"] = df["world_rank"].apply(lambda x: createCateg(x))

df.head()

,world_rank,institution,country,national_rank,quality_of_education,alumni_employment,quality_of_faculty,publications,influence,citations,broad_impact,patents,score,year,Rank_Level
0,1,Harvard University,USA,1,7,9,1,1,1,1,NaN,5,100.00,2012,First Tier Top University
1,2,Massachusetts Institute of Technology,USA,2,9,17,3,12,4,4,NaN,1,91.67,2012,First Tier Top University
2,3,Stanford University,USA,3,17,11,5,4,2,2,NaN,15,89.50,2012,First Tier Top University
3,4,University of Cambridge,United Kingdom,1,10,24,4,16,16,11,NaN,50,86.17,2012,First Tier Top University
4,5,California Institute of Technology,USA,4,2,29,7,37,22,22,NaN,18,85.21,2012,First Tier Top University


### Creating a Pivot Table

Now we create a pivot table using `df.pivot_table()`. We compare countries (index) against university tiers (columns) and aggregate by the mean of their scores (values).

In [12]:
df.pivot_table(values='score', index='country', columns='Rank_Level', aggfunc=[np.nanmean]).head()

C:\Users\kanko\AppData\Local\Temp\ipykernel_12220\148465126.py:1: FutureWarning: The provided callable <function nanmean at 0x000001FA6078F600> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df.pivot_table(values='score', index='country', columns='Rank_Level', aggfunc=[np.nanmean]).head()


nanmean                       \
Rank_Level First Tier Top University Other Top University   
country                                                     
Argentina                        NaN            44.672857   
Australia                    47.9425            44.645750   
Austria                          NaN            44.864286   
Belgium                      51.8750            45.081000   
Brazil                           NaN            44.499706   

                                                                 
Rank_Level Second Tier Top University Third Tier Top University  
country                                                          
Argentina                         NaN                       NaN  
Australia                     49.2425                 47.285000  
Austria                           NaN                 47.066667  
Belgium                       49.0840                 46.746667  
Brazil                        49.5650                       NaN

We can pass a list of functions to `aggfunc` to calculate multiple aggregate values simultaneously (e.g., `np.mean` and `np.max`). This creates a hierarchical column index.

In [13]:
df.pivot_table(values='score', index='country', columns='Rank_Level', aggfunc=[np.mean, np.max]).head()

C:\Users\kanko\AppData\Local\Temp\ipykernel_12220\2535016970.py:1: FutureWarning: The provided callable <function mean at 0x000001FA6064B420> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df.pivot_table(values='score', index='country', columns='Rank_Level', aggfunc=[np.mean, np.max]).head()
C:\Users\kanko\AppData\Local\Temp\ipykernel_12220\2535016970.py:1: FutureWarning: The provided callable <function max at 0x000001FA6064AA20> is currently using DataFrameGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  df.pivot_table(values='score', index='country', columns='Rank_Level', aggfunc=[np.mean, np.max]).head()


mean                       \
Rank_Level First Tier Top University Other Top University   
country                                                     
Argentina                        NaN            44.672857   
Australia                    47.9425            44.645750   
Austria                          NaN            44.864286   
Belgium                      51.8750            45.081000   
Brazil                           NaN            44.499706   

                                                                 \
Rank_Level Second Tier Top University Third Tier Top University   
country                                                           
Argentina                         NaN                       NaN   
Australia                     49.2425                 47.285000   
Austria                           NaN                 47.066667   
Belgium                       49.0840                 46.746667   
Brazil                        49.5650                       NaN   

                                 max                       \
Rank_Level First Tier Top University Other Top University   
country                                                     
Argentina                        NaN                45.66   
Australia                      51.61                45.97   
Austria                          NaN                46.29   
Belgium                        52.03                46.21   
Brazil                           NaN                46.08   

                                                                 
Rank_Level Second Tier Top University Third Tier Top University  
country                                                          
Argentina                         NaN                       NaN  
Australia                       50.40                     47.47  
Austria                           NaN                     47.78  
Belgium                         49.73                     47.14  
Brazil                          49.82                       NaN

If we set `margins=True`, pandas will add an 'All' row and column, calculating the overall aggregate across all groups.

In [14]:
df.pivot_table(values='score', index='country', columns='Rank_Level', aggfunc=[np.mean, np.max], 
               margins=True).head()

C:\Users\kanko\AppData\Local\Temp\ipykernel_12220\712757933.py:1: FutureWarning: The provided callable <function mean at 0x000001FA6064B420> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df.pivot_table(values='score', index='country', columns='Rank_Level', aggfunc=[np.mean, np.max],
C:\Users\kanko\AppData\Local\Temp\ipykernel_12220\712757933.py:1: FutureWarning: The provided callable <function mean at 0x000001FA6064B420> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df.pivot_table(values='score', index='country', columns='Rank_Level', aggfunc=[np.mean, np.max],
C:\Users\kanko\AppData\Local\Temp\ipykernel_12220\712757933.py:1: FutureWarning: The provided callable <function mean at 0x000001FA6064B420> is currently using DataFrameG

mean                       \
Rank_Level First Tier Top University Other Top University   
country                                                     
Argentina                        NaN            44.672857   
Australia                    47.9425            44.645750   
Austria                          NaN            44.864286   
Belgium                      51.8750            45.081000   
Brazil                           NaN            44.499706   

                                                                            \
Rank_Level Second Tier Top University Third Tier Top University        All   
country                                                                      
Argentina                         NaN                       NaN  44.672857   
Australia                     49.2425                 47.285000  45.825517   
Austria                           NaN                 47.066667  45.139583   
Belgium                       49.0840                 46.746667  47.011000   
Brazil                        49.5650                       NaN  44.781111   

                                 max                       \
Rank_Level First Tier Top University Other Top University   
country                                                     
Argentina                        NaN                45.66   
Australia                      51.61                45.97   
Austria                          NaN                46.29   
Belgium                        52.03                46.21   
Brazil                           NaN                46.08   

                                                                        
Rank_Level Second Tier Top University Third Tier Top University    All  
country                                                                 
Argentina                         NaN                       NaN  45.66  
Australia                       50.40                     47.47  51.61  
Austria                           NaN                     47.78  47.78  
Belgium                         49.73                     47.14  52.03  
Brazil                          49.82                       NaN  49.82

Let's save this `margins=True` pivot table into a new variable `new_df` so we can explore its structure.

In [15]:
new_df=df.pivot_table(values='score', index='country', columns='Rank_Level', aggfunc=[np.mean, np.max], 
               margins=True)

print(new_df.index)

print(new_df.columns)

Index(['Argentina', 'Australia', 'Austria', 'Belgium', 'Brazil', 'Bulgaria',
       'Canada', 'Chile', 'China', 'Colombia', 'Croatia', 'Cyprus',
       'Czech Republic', 'Denmark', 'Egypt', 'Estonia', 'Finland', 'France',
       'Germany', 'Greece', 'Hong Kong', 'Hungary', 'Iceland', 'India', 'Iran',
       'Ireland', 'Israel', 'Italy', 'Japan', 'Lebanon', 'Lithuania',
       'Malaysia', 'Mexico', 'Netherlands', 'New Zealand', 'Norway', 'Poland',
       'Portugal', 'Puerto Rico', 'Romania', 'Russia', 'Saudi Arabia',
       'Serbia', 'Singapore', 'Slovak Republic', 'Slovenia', 'South Africa',
       'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Taiwan', 'Thailand',
       'Turkey', 'USA', 'Uganda', 'United Arab Emirates', 'United Kingdom',
       'Uruguay', 'All'],
      dtype='object', name='country')
MultiIndex([('mean',  'First Tier Top University'),
            ('mean',       'Other Top University'),
            ('mean', 'Second Tier Top University'),
            ('mean',  'Thir

C:\Users\kanko\AppData\Local\Temp\ipykernel_12220\637163146.py:1: FutureWarning: The provided callable <function mean at 0x000001FA6064B420> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  new_df=df.pivot_table(values='score', index='country', columns='Rank_Level', aggfunc=[np.mean, np.max],
C:\Users\kanko\AppData\Local\Temp\ipykernel_12220\637163146.py:1: FutureWarning: The provided callable <function mean at 0x000001FA6064B420> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  new_df=df.pivot_table(values='score', index='country', columns='Rank_Level', aggfunc=[np.mean, np.max],
C:\Users\kanko\AppData\Local\Temp\ipykernel_12220\637163146.py:1: FutureWarning: The provided callable <function mean at 0x000001FA6064B420> is currently us

### Querying Pivot Tables

Because the columns are hierarchical (MultiIndex), to extract a column series we use tuple-like chaining: first 'mean', then 'First Tier Top University'.

In [17]:
new_df['mean']['First Tier Top University'].head()

country
Argentina        NaN
Australia    47.9425
Austria          NaN
Belgium      51.8750
Brazil           NaN
Name: First Tier Top University, dtype: float64

In [19]:
type(new_df['mean']['First Tier Top University'])

pandas.core.series.Series

We can use `.idxmax()` to find the index (country) with the maximum value in that specific tier/aggregation combination. Here, the UK has the highest average score for First Tier universities.

In [20]:
new_df['mean']['First Tier Top University'].idxmax()

'United Kingdom'

### Stack and Unstack

The shape of pivot tables can be modified using `.stack()` and `.unstack()`. Here is our original `new_df` shape: 

In [21]:
new_df.head()

mean                       \
Rank_Level First Tier Top University Other Top University   
country                                                     
Argentina                        NaN            44.672857   
Australia                    47.9425            44.645750   
Austria                          NaN            44.864286   
Belgium                      51.8750            45.081000   
Brazil                           NaN            44.499706   

                                                                            \
Rank_Level Second Tier Top University Third Tier Top University        All   
country                                                                      
Argentina                         NaN                       NaN  44.672857   
Australia                     49.2425                 47.285000  45.825517   
Austria                           NaN                 47.066667  45.139583   
Belgium                       49.0840                 46.746667  47.011000   
Brazil                        49.5650                       NaN  44.781111   

                                 max                       \
Rank_Level First Tier Top University Other Top University   
country                                                     
Argentina                        NaN                45.66   
Australia                      51.61                45.97   
Austria                          NaN                46.29   
Belgium                        52.03                46.21   
Brazil                           NaN                46.08   

                                                                        
Rank_Level Second Tier Top University Third Tier Top University    All  
country                                                                 
Argentina                         NaN                       NaN  45.66  
Australia                       50.40                     47.47  51.61  
Austria                           NaN                     47.78  47.78  
Belgium                         49.73                     47.14  52.03  
Brazil                          49.82                       NaN  49.82

`.stack()` pivots the innermost column index down into a new innermost row index, making the DataFrame longer and narrower.

In [22]:
new_df = new_df.stack()
new_df

C:\Users\kanko\AppData\Local\Temp\ipykernel_12220\2881191465.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  new_df = new_df.stack()


mean     max
country   Rank_Level                                   
Argentina Other Top University        44.672857   45.66
          All                         44.672857   45.66
Australia First Tier Top University   47.942500   51.61
          Other Top University        44.645750   45.97
          Second Tier Top University  49.242500   50.40
...                                         ...     ...
All       First Tier Top University   58.350675  100.00
          Other Top University        44.738871   46.34
          Second Tier Top University  49.065450   51.29
          Third Tier Top University   46.843450   47.93
          All                         47.798395  100.00

[193 rows x 2 columns]

`.unstack()` does the opposite; it pivots the innermost row index up into a new innermost column index. Since our rows were countries, the columns are now country/tier/aggfunc combinations.

In [23]:
new_df.unstack().head()

mean                       \
Rank_Level First Tier Top University Other Top University   
country                                                     
All                        58.350675            44.738871   
Argentina                        NaN            44.672857   
Australia                  47.942500            44.645750   
Austria                          NaN            44.864286   
Belgium                    51.875000            45.081000   

                                                                            \
Rank_Level Second Tier Top University Third Tier Top University        All   
country                                                                      
All                          49.06545                 46.843450  47.798395   
Argentina                         NaN                       NaN  44.672857   
Australia                    49.24250                 47.285000  45.825517   
Austria                           NaN                 47.066667  45.139583   
Belgium                      49.08400                 46.746667  47.011000   

                                 max                       \
Rank_Level First Tier Top University Other Top University   
country                                                     
All                           100.00                46.34   
Argentina                        NaN                45.66   
Australia                      51.61                45.97   
Austria                          NaN                46.29   
Belgium                        52.03                46.21   

                                                                         
Rank_Level Second Tier Top University Third Tier Top University     All  
country                                                                  
All                             51.29                     47.93  100.00  
Argentina                         NaN                       NaN   45.66  
Australia                       50.40                     47.47   51.61  
Austria                           NaN                     47.78   47.78  
Belgium                         49.73                     47.14   52.03

If we `.unstack()` twice, we flip the entire structure recursively, ending up with a Series if we unstack all row indices.

In [24]:
new_df.unstack().unstack().head()

      Rank_Level                 country  
mean  First Tier Top University  All          58.350675
                                 Argentina          NaN
                                 Australia    47.942500
                                 Austria            NaN
                                 Belgium      51.875000
dtype: float64